In [ ]:
import pandas as pd

# Charge le fichier brut (".." remonte d'un dossier, de notebooks vers data_brute)
df = pd.read_csv("../data_brute/ctg-studies.csv")

# Nombre de lignes (études) et de colonnes
print(df.shape)

In [ ]:
# Affiche la liste de toutes les colonnes, une par ligne
for colonne in df.columns:
    print(colonne)

In [ ]:
# Affiche les lieux de la 1re étude (la colonne peut être très longue)
print(df["Locations"].iloc[0])

# Affiche les lieux de la 2e étude
print(df["Locations"].iloc[1])

In [ ]:
# Fonction : prend le texte des lieux d'UNE étude et renvoie la liste de ses pays
def extraire_pays(texte):
    # Si la case est vide (étude sans lieu), on renvoie une liste vide
    if pd.isna(texte):
        return []
    # On coupe le texte à chaque "|" : on obtient un lieu par morceau
    lieux = texte.split("|")
    # Pour chaque lieu, on coupe aux virgules et on garde le DERNIER morceau (le pays)
    pays = [lieu.split(",")[-1].strip() for lieu in lieux]
    # set() enlève les doublons (un pays compté une fois par étude), sorted() les range
    return sorted(set(pays))

# On applique la fonction à chaque ligne et on range le résultat dans une nouvelle colonne
df["Pays"] = df["Locations"].apply(extraire_pays)

# On regarde les 5 premières études avec leurs pays
print(df[["NCT Number", "Pays"]].head(5))

# On compte les études sans aucun lieu renseigné
print("Études sans pays :", (df["Pays"].str.len() == 0).sum())

In [ ]:
# Combien d'études ont des résultats déposés ?
print(df["Study Results"].value_counts())

# Répartition des études par statut
print(df["Study Status"].value_counts())

In [ ]:
# Les 98 études sans pays : quel est leur statut ?
sans_pays = df[df["Pays"].str.len() == 0]
print(sans_pays["Study Status"].value_counts())

# Combien d'études par pays (une étude compte une seule fois par pays)
from collections import Counter
compteur = Counter(p for liste in df["Pays"] for p in liste)
for pays, n in compteur.most_common(25):
    print(pays, n)

# Nombre de pays différents trouvés
print("Nombre de pays différents :", len(compteur))

In [ ]:
# Liste complète des pays, par ordre alphabétique, avec le nombre d'études
for pays in sorted(compteur):
    print(pays, "-", compteur[pays])

In [ ]:
# Retrouve l'étude concernée
ligne = df[df["Locations"].str.contains("Arthur M. Blank", na=False)]

# Affiche uniquement le lieu en question, et le numéro de l'étude
for texte in ligne["Locations"]:
    for lieu in texte.split("|"):
        if "Arthur M. Blank" in lieu:
            print(lieu)
print(ligne["NCT Number"].tolist())

In [ ]:
from collections import Counter

# Table de corrections : "mauvais nom" -> "bon nom"
corrections = {
    "Arthur M. Blank Hospital": "United States",
    "Turkey (Türkiye)": "Turkey",
}

# Remplace les mauvais noms dans la liste de pays de chaque étude
def corriger_pays(liste):
    nouveaux = [corrections.get(p, p) for p in liste]
    return sorted(set(nouveaux))

df["Pays"] = df["Pays"].apply(corriger_pays)

# On recompte les études par pays
compteur = Counter(p for liste in df["Pays"] for p in liste)

# Vérifications
print("Nombre de pays différents :", len(compteur))
print("United States :", compteur["United States"])
print("Turkey :", compteur["Turkey"])
print("Arthur M. Blank Hospital présent ?", "Arthur M. Blank Hospital" in compteur)

In [ ]:
# Table de correspondance : pays -> région
regions = {
    "Afrique subsaharienne": ["Angola", "Burkina Faso", "Cameroon", "Côte d’Ivoire",
        "Democratic Republic of the Congo", "Gabon", "Ghana", "Kenya", "Malawi", "Mali",
        "Nigeria", "Reunion", "Senegal", "Sierra Leone", "South Africa", "Tanzania",
        "Uganda", "Zambia"],
    "Afrique du Nord": ["Egypt", "Morocco", "Sudan", "Tunisia"],
    "Moyen-Orient": ["Bahrain", "Iraq", "Israel", "Jordan", "Kuwait", "Lebanon", "Oman",
        "Qatar", "Saudi Arabia", "Turkey", "United Arab Emirates"],
    "Europe": ["Austria", "Belgium", "Cyprus", "Czechia", "Denmark", "Finland", "France",
        "Germany", "Greece", "Ireland", "Italy", "Netherlands", "Poland", "Portugal",
        "Russia", "Spain", "Switzerland", "United Kingdom"],
    "Amérique du Nord": ["Canada", "United States"],
    "Caraïbes et Amérique latine": ["Argentina", "Brazil", "Colombia", "Dominican Republic",
        "French Guiana", "Guadeloupe", "Honduras", "Jamaica", "Martinique", "Panama",
        "Puerto Rico", "Trinidad and Tobago"],
    "Asie et Océanie": ["Australia", "Bangladesh", "Brunei", "China", "Hong Kong", "India",
        "Japan", "Malaysia", "Nepal", "Pakistan", "South Korea", "Taiwan", "Thailand"],
}

# On inverse la table : pays -> région (plus pratique pour chercher)
pays_vers_region = {}
for region, liste_pays in regions.items():
    for p in liste_pays:
        pays_vers_region[p] = region

# Vérification : y a-t-il des pays de notre liste sans région ?
manquants = [p for p in compteur if p not in pays_vers_region]
print("Pays sans région :", manquants)

# Pour chaque étude, la liste de ses régions (sans doublons)
df["Regions"] = df["Pays"].apply(
    lambda liste: sorted(set(pays_vers_region[p] for p in liste if p in pays_vers_region))
)

# Nombre d'études par région
compteur_regions = Counter(r for liste in df["Regions"] for r in liste)
for region, n in compteur_regions.most_common():
    print(region, "-", n)

# Études avec au moins un site en Afrique (subsaharienne ou du Nord)
afrique = df["Regions"].apply(
    lambda l: "Afrique subsaharienne" in l or "Afrique du Nord" in l
)
print("Études avec au moins un site en Afrique :", afrique.sum())

In [ ]:
# Phases des essais (une étude peut avoir plusieurs phases, ex. "PHASE1|PHASE2")
print(df["Phases"].value_counts(dropna=False))

# Type de financeur
print(df["Funder Type"].value_counts(dropna=False))

# Les 15 sponsors les plus fréquents
print(df["Sponsor"].value_counts().head(15))

In [ ]:
# Type d'étude (interventionnelle ou observationnelle)
print(df["Study Type"].value_counts(dropna=False))

# Pour les études sans phase : quel type d'étude ?
print(df[df["Phases"].isna()]["Study Type"].value_counts(dropna=False))

# Exemples de la colonne Interventions (3 études qui ont une phase)
for texte in df[df["Phases"].notna()]["Interventions"].head(3):
    print(texte)
    print("---")

In [ ]:
from collections import Counter

# Pour chaque étude, on extrait les types d'intervention (le mot avant ":")
def extraire_types(texte):
    if pd.isna(texte):
        return []
    return sorted(set(i.split(":")[0].strip() for i in texte.split("|")))

df["Types_intervention"] = df["Interventions"].apply(extraire_types)

# Combien d'études par type
compteur_types = Counter(t for liste in df["Types_intervention"] for t in liste)
print(compteur_types.most_common())

# Les 40 noms de DRUG et GENETIC les plus fréquents
noms = []
for texte in df["Interventions"].dropna():
    for i in texte.split("|"):
        if i.startswith("DRUG:") or i.startswith("GENETIC:"):
            noms.append(i.strip())
for nom, n in Counter(noms).most_common(40):
    print(nom, "-", n)

In [ ]:
# Dictionnaire : famille -> mots-clés (en minuscules)
familles = {
    "Hydroxyurée": ["hydroxyurea"],
    "Nouveaux médicaments ciblés": ["voxelotor", "gbt440", "gbt021601", "mitapivat",
        "etavopivat", "imr-687", "crizanlizumab", "aes-103"],
    "Greffe de moelle (conditionnement)": ["alemtuzumab", "sirolimus", "fludarabine",
        "cyclophosphamide", "busulfan", "tacrolimus", "thiotepa", "melphalan",
        "mycophenolate", "cyclosporine", "rituximab", "abatacept", "methotrexate", "mesna"],
    "Mobilisation de cellules souches": ["plerixafor", "motixafortide", "filgrastim"],
    "Douleur": ["ketamine", "morphine"],
    "Complications": ["citrulline", "arginine", "l-nmma", "nitric oxide", "prasugrel",
        "losartan", "deferasirox"],
    "Placebo / contrôle": ["placebo", "saline"],
}

# Donne la famille d'un nom de médicament (en ignorant majuscules/minuscules)
def classer_nom(nom):
    nom = nom.lower()
    for famille, mots in familles.items():
        if any(mot in nom for mot in mots):
            return famille
    return "À classer"

# Liste des familles pour UNE étude (on ne regarde que les lignes "DRUG:")
def familles_etude(texte):
    if pd.isna(texte):
        return []
    fams = set()
    for i in texte.split("|"):
        if i.startswith("DRUG:"):
            fams.add(classer_nom(i.replace("DRUG:", "").strip()))
    return sorted(fams)

df["Familles_medicaments"] = df["Interventions"].apply(familles_etude)

# Nombre d'études par famille
c = Counter(f for liste in df["Familles_medicaments"] for f in liste)
for famille, n in c.most_common():
    print(famille, "-", n)

# Les médicaments qui restent "à classer" (40 plus fréquents)
restants = Counter()
for texte in df["Interventions"].dropna():
    for i in texte.split("|"):
        if i.startswith("DRUG:"):
            nom = i.replace("DRUG:", "").strip()
            if classer_nom(nom) == "À classer":
                restants[nom.lower()] += 1
for nom, n in restants.most_common(40):
    print(nom, "-", n)

In [ ]:
# Dictionnaire complété : famille -> mots-clés (en minuscules)
familles = {
    "Hydroxyurée": ["hydroxyurea", "hydroxycarbamide", "siklos", "droxia", "hydrea"],
    "Nouveaux médicaments ciblés": ["voxelotor", "gbt440", "gbt021601", "mitapivat",
        "etavopivat", "imr-687", "pde9i", "crizanlizumab", "aes-103", "l-glutamine",
        "hqk-1001", "inclacumab", "pociredir", "ica-17043", "osivelotor", "rivipansel",
        "gmi-1070", "crovalimab", "hbi-002"],
    "Greffe de moelle (conditionnement)": ["alemtuzumab", "sirolimus", "fludarabine",
        "cyclophosphamide", "busulfan", "tacrolimus", "thiotepa", "melphalan",
        "mycophenolate", "cyclosporine", "rituximab", "abatacept", "methotrexate",
        "mesna", "pentostatin", "thymoglobulin", "atg", "bortezomib"],
    "Mobilisation de cellules souches": ["plerixafor", "motixafortide", "filgrastim"],
    "Douleur": ["ketamine", "morphine", "ibuprofen"],
    "Chélation du fer": ["deferasirox", "deferoxamine", "deferiprone"],
    "Vasculaire et cardio-pulmonaire": ["citrulline", "arginine", "l-nmma", "nitric oxide",
        "prasugrel", "losartan", "ticagrelor", "aspirin", "sildenafil", "bosentan",
        "acetylcholine", "lexiscan", "atorvastatin", "simvastatin", "defibrotide",
        "isoquercetin"],
    "Antioxydants, vitamines et suppléments": ["n-acetylcysteine", "zinc", "vitamin d"],
    "Anti-infectieux": ["azithromycin"],
    "Placebo / contrôle": ["placebo", "saline", "standard of care"],
}

# On relance le classement avec le nouveau dictionnaire
df["Familles_medicaments"] = df["Interventions"].apply(familles_etude)

# Nombre d'études par famille
c = Counter(f for liste in df["Familles_medicaments"] for f in liste)
for famille, n in c.most_common():
    print(famille, "-", n)

# Les 40 médicaments qui restent "à classer"
restants = Counter()
for texte in df["Interventions"].dropna():
    for i in texte.split("|"):
        if i.startswith("DRUG:"):
            nom = i.replace("DRUG:", "").strip()
            if classer_nom(nom) == "À classer":
                restants[nom.lower()] += 1
for nom, n in restants.most_common(40):
    print(nom, "-", n)

In [ ]:
# On AJOUTE des mots-clés aux familles existantes (le dictionnaire "familles" doit exister)
familles["Hydroxyurée"] += ["xromi"]
familles["Placebo / contrôle"] += ["sodium chloride", "nacl"]
familles["Greffe de moelle (conditionnement)"] += ["fk506", "campath",
    "anti-thymocyte", "siplizumab", "azathioprine", "rimiducid", "apamistamab"]
familles["Douleur"] += ["fentanyl", "sufentanil", "acetaminophen", "midazolam"]
familles["Vasculaire et cardio-pulmonaire"] += ["rosuvastatin", "tadalafil",
    "riociguat", "regadenoson"]
familles["Nouveaux médicaments ciblés"] += ["glutamine", "senicapoc", "mst-188"]

# Nouvelle famille
familles["Corticoïdes"] = ["dexamethasone", "methylprednisolone"]

# On relance le classement
df["Familles_medicaments"] = df["Interventions"].apply(familles_etude)

# Nombre d'études par famille
c = Counter(f for liste in df["Familles_medicaments"] for f in liste)
for famille, n in c.most_common():
    print(famille, "-", n)

# Études dont TOUS les médicaments restent non classés
seulement_a_classer = df["Familles_medicaments"].apply(lambda l: l == ["À classer"])
print("Études uniquement 'À classer' :", seulement_a_classer.sum())

# Études sans aucun médicament (comportement, dispositif, observationnel...)
print("Études sans médicament :", (df["Familles_medicaments"].str.len() == 0).sum())

In [ ]:
# On renomme la catégorie "À classer" (relancer la cellule de classement la remettrait à l'ancien nom)
df["Familles_medicaments"] = df["Familles_medicaments"].apply(
    lambda l: ["Autres / à vérifier" if f == "À classer" else f for f in l]
)

# Sous-ensemble : les études qui ont des résultats déposés
res = df[df["Study Results"] == "YES"]
print("Études avec résultats :", len(res))

# Leur statut
print(res["Study Status"].value_counts())

# Leur type d'étude
print(res["Study Type"].value_counts())

# Leurs phases
print(res["Phases"].value_counts(dropna=False))

# Leurs familles de médicaments
c_res = Counter(f for liste in res["Familles_medicaments"] for f in liste)
for famille, n in c_res.most_common():
    print(famille, "-", n)

In [ ]:
import json
import urllib.request

# On prend une étude ARRÊTÉE qui a des résultats (pour voir les deux informations)
exemple = df[(df["Study Status"] == "TERMINATED") & (df["Study Results"] == "YES")]
nct = exemple["NCT Number"].iloc[0]
print("Étude testée :", nct)

# Adresse de l'API pour cette étude
url = f"https://clinicaltrials.gov/api/v2/studies/{nct}"
requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

# On télécharge et on lit la réponse (au format JSON)
with urllib.request.urlopen(requete) as reponse:
    donnees = json.loads(reponse.read())

# Les grandes sections disponibles
print("Sections :", list(donnees.keys()))

# Le motif d'arrêt
statut = donnees["protocolSection"]["statusModule"]
print("Statut :", statut.get("overallStatus"))
print("Motif d'arrêt :", statut.get("whyStopped"))

# Les sections des résultats
if "resultsSection" in donnees:
    print("Résultats :", list(donnees["resultsSection"].keys()))
    mesures = donnees["resultsSection"]["outcomeMeasuresModule"]["outcomeMeasures"]
    print("Nombre de critères mesurés :", len(mesures))
    print("Premier critère :", mesures[0]["title"])

In [ ]:
import os
import time
import json
import urllib.request

# Dossier où on range les fiches (une par étude)
dossier = "../data_brute/api_json"
os.makedirs(dossier, exist_ok=True)

# Les études à télécharger : arrêtées/retirées OU avec résultats
arretees = df[df["Study Status"].isin(["TERMINATED", "WITHDRAWN", "SUSPENDED"])]["NCT Number"]
avec_resultats = df[df["Study Results"] == "YES"]["NCT Number"]
a_telecharger = sorted(set(arretees) | set(avec_resultats))
print("Études à télécharger :", len(a_telecharger))

erreurs = []
for i, nct in enumerate(a_telecharger, start=1):
    fichier = os.path.join(dossier, f"{nct}.json")

    # Si la fiche est déjà sur l'ordinateur, on passe (permet de reprendre après une coupure)
    if os.path.exists(fichier):
        continue

    try:
        url = f"https://clinicaltrials.gov/api/v2/studies/{nct}"
        requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(requete, timeout=30) as reponse:
            donnees = json.loads(reponse.read())
        with open(fichier, "w", encoding="utf-8") as f:
            json.dump(donnees, f)
    except Exception as e:
        erreurs.append((nct, str(e)))

    # Petite pause pour ne pas surcharger le serveur
    time.sleep(0.3)

    # Un message de progression toutes les 50 études
    if i % 50 == 0:
        print(f"{i} / {len(a_telecharger)}")

print("Terminé.")
print("Fiches sur l'ordinateur :", len(os.listdir(dossier)))
print("Erreurs :", len(erreurs))
for nct, msg in erreurs[:5]:
    print(nct, msg)

In [ ]:
import json
import os

dossier = "../data_brute/api_json"

# Lit le motif d'arrêt dans la fiche d'une étude (renvoie None si absent)
def lire_motif(nct):
    fichier = os.path.join(dossier, f"{nct}.json")
    if not os.path.exists(fichier):
        return None
    with open(fichier, encoding="utf-8") as f:
        donnees = json.load(f)
    return donnees["protocolSection"]["statusModule"].get("whyStopped")

# Les études arrêtées ou retirées
arretes = df[df["Study Status"].isin(["TERMINATED", "WITHDRAWN", "SUSPENDED"])].copy()
arretes["Motif_arret"] = arretes["NCT Number"].apply(lire_motif)

# Combien ont un motif, combien n'en ont pas
print("Études arrêtées ou retirées :", len(arretes))
print("Avec un motif :", arretes["Motif_arret"].notna().sum())
print("Sans motif :", arretes["Motif_arret"].isna().sum())

# Même chose, par statut
print(arretes.groupby("Study Status")["Motif_arret"].apply(lambda s: s.notna().sum()))

# Tous les motifs, par ordre alphabétique (coupés à 120 caractères)
for motif in arretes["Motif_arret"].dropna().sort_values():
    print("-", motif[:120])

In [ ]:
import re
from collections import Counter

# Catégories -> mots-clés à repérer dans le motif (en minuscules)
categories_motifs = {
    "Sécurité / risque pour les patients": ["safety", "risk profile", "from the market",
        "more likely to have severe pain"],
    "Conception de l'étude / données peu informatives": ["reliable endpoint", "informative",
        "unblinding", "unblinded", "draw conclusions", "baseline pain score", "variab"],
    "Efficacité insuffisante / futilité": ["efficacy", "futility", "low probability", "failed"],
    "Objectif atteint / données suffisantes": ["successfully meeting", "sufficient data",
        "sufficient information", "results already published", "completed at site",
        "enrollment completion"],
    "Recommandation d'un comité de surveillance": ["dsmb", "data monitoring committee", "bsmb"],
    "COVID-19": ["covid", "pandemic", "coronavirus"],
    "Recrutement insuffisant": ["enrol", "accru", "recruit", "inclusion", "eligible",
        "paucity", "sample size", "pool of potential", "participants", "no patients",
        "never started", "target number", "unable to identify", "interest in study"],
    "Financement": ["fund", "cost", "resources"],
    "Décision du promoteur ou de l'investigateur": ["sponsor", "business", "company",
        "promoter", "industry collaborator", "strategic", "portfolio", "development plan",
        "program", "liquidation", "ceased operations", "no longer required",
        "direction of", "terminated by", "decided not", "no intention", "cancelled",
        "not a clinical trial", "decision of"],
    "Évolution du contexte (concurrence, standard de soins)": ["competing", "standard of care",
        "replaced by", "preference given", "more effective", "no longer needed",
        "initiation of cms"],
    "Logistique, personnel ou approvisionnement": [r"\bpi\b", "principal investigator",
        "key personnel", "relocation", "moved", "moving", "leaving", "returned to",
        "equipment", "manufacturing", "supplies", "medicine", "no longer making",
        "technical difficulties", "unable to secure", "ips lines"],
}

# Motifs classés à la main (début du texte -> catégorie), car les mots-clés se tromperaient
corrections_manuelles = {
    "the safety and efficacy of plerixafor": "Évolution du contexte (concurrence, standard de soins)",
}

# Retire les phrases du type "not related to safety" avant de chercher "safety"
def retirer_negations_securite(texte):
    return re.sub(r"\b(not|no)\b[^.]{0,40}?safety", " ", texte)

# Renvoie la liste des catégories d'un motif
def classer_motif(motif):
    if pd.isna(motif):
        return ["Motif non renseigné"]
    t = motif.lower()
    for debut, cat in corrections_manuelles.items():
        if t.startswith(debut):
            return [cat]
    t_secu = retirer_negations_securite(t)
    trouvees = []
    for cat, mots in categories_motifs.items():
        texte = t_secu if cat.startswith("Sécurité") else t
        if any(re.search(m, texte) for m in mots):
            trouvees.append(cat)
    return trouvees if trouvees else ["Autre / à vérifier"]

arretes["Categories_arret"] = arretes["Motif_arret"].apply(classer_motif)

# Nombre d'études par catégorie (une étude peut compter dans plusieurs)
c = Counter(cat for liste in arretes["Categories_arret"] for cat in liste)
for cat, n in c.most_common():
    print(cat, "-", n)

# Études avec au moins 2 catégories
print("Études à plusieurs catégories :", (arretes["Categories_arret"].str.len() > 1).sum())

# Catégories selon le statut (terminée après démarrage / retirée avant inclusion)
expl = arretes.explode("Categories_arret")
print(pd.crosstab(expl["Categories_arret"], expl["Study Status"]))

# Les motifs que le code n'a pas su classer, en entier
print("\nMotifs à vérifier :")
for motif in arretes[arretes["Categories_arret"].apply(lambda l: l == ["Autre / à vérifier"])]["Motif_arret"]:
    print("-", motif)

In [ ]:
# Corrections faites à la main : motif exact -> catégorie
manuel = {
    "Data will not inform further development of Voxelotor":
        ["Décision du promoteur ou de l'investigateur"],
    "Feasibility": ["Logistique, personnel ou approvisionnement"],
    "Unanticipated delays, not for safety reasons":
        ["Logistique, personnel ou approvisionnement"],
    "Trial closed just prior to study receiving Investigational Review Board approval":
        ["Logistique, personnel ou approvisionnement"],
    "Participant withdrew consent": ["Recrutement insuffisant"],
    "Study on hold pending consideration for amendment.":
        ["Conception de l'étude / données peu informatives"],
}

# Applique la correction si le motif est dans la liste, sinon garde la catégorie du code
def appliquer_manuel(ligne):
    motif = ligne["Motif_arret"]
    if pd.notna(motif) and motif.strip() in manuel:
        return manuel[motif.strip()]
    return ligne["Categories_arret"]

arretes["Categories_arret"] = arretes.apply(appliquer_manuel, axis=1)

# Résultat final : études par catégorie et part sur 182
total = len(arretes)
c = Counter(cat for liste in arretes["Categories_arret"] for cat in liste)
for cat, n in c.most_common():
    print(f"{cat} - {n} ({n / total:.0%})")

# L'hypothèse "décision du promoteur = argent", chiffrée sans fusionner les catégories
a_fin = arretes["Categories_arret"].apply(lambda l: "Financement" in l)
a_dec = arretes["Categories_arret"].apply(
    lambda l: "Décision du promoteur ou de l'investigateur" in l)
print("\nFinancement seul :", a_fin.sum())
print("Financement ou décision du promoteur :", (a_fin | a_dec).sum())

In [ ]:
# Lit les critères principaux (définis dans le protocole) d'une étude
def lire_criteres_principaux(nct):
    fichier = os.path.join(dossier, f"{nct}.json")
    with open(fichier, encoding="utf-8") as f:
        donnees = json.load(f)
    criteres = donnees["protocolSection"].get("outcomesModule", {}).get("primaryOutcomes", [])
    return [c.get("measure", "") for c in criteres]

res = df[df["Study Results"] == "YES"].copy()
res["Criteres_principaux"] = res["NCT Number"].apply(lire_criteres_principaux)

# Mots-clés par thème (en minuscules)
themes = {
    "Douleur / crises": ["pain", "crisis", "crises", "voc", "vaso-occlusive"],
    "Hémoglobine / hémolyse": ["hemoglobin", "haemoglobin", "hemolysis", "hemolytic", "hbf"],
    "Hospitalisation": ["hospital", "admission"],
    "Qualité de vie / symptômes": ["quality of life", "fatigue", "questionnaire", "pedsql"],
    "Greffe / survie": ["engraftment", "survival", "graft", "chimerism", "rejection"],
    "Cerveau / vaisseaux": ["stroke", "transcranial", "doppler", "velocity", "cerebral", "infarct"],
    "Organes (rein, poumon, cœur)": ["renal", "kidney", "gfr", "albumin", "pulmonary", "lung", "cardiac"],
    "Sécurité / effets indésirables": ["adverse", "safety", "toxicity", "tolerab"],
    "Pharmacologie (taux sanguins)": ["pharmacokinetic", "concentration", "auc", "cmax"],
}

# Pour chaque thème : combien d'études ont au moins un critère principal qui le mentionne
for theme, mots in themes.items():
    n = res["Criteres_principaux"].apply(
        lambda liste: any(m in c.lower() for c in liste for m in mots)
    ).sum()
    print(f"{theme} - {n}")

# Études dont aucun critère principal ne correspond à un thème
def aucun_theme(liste):
    return not any(m in c.lower() for c in liste for mots in themes.values() for m in mots)
sans_theme = res[res["Criteres_principaux"].apply(aucun_theme)]
print("\nÉtudes sans thème reconnu :", len(sans_theme))

# Un échantillon de leurs critères, pour compléter la liste de mots-clés
for liste in sans_theme["Criteres_principaux"].head(40):
    print("-", " | ".join(liste)[:130])

In [ ]:
import re

# Thèmes -> expressions à repérer (en minuscules). \b = début ou fin de mot
themes = {
    "Douleur / crises": [r"pain", r"crisis", r"crises", r"\bvoc\b", r"vaso-occlusive"],
    "Hémoglobine / hémolyse": [r"hemoglobin", r"haemoglobin", r"\bhb", r"hemoly",
        r"haemoly", r"bilirubin", r"carbon monoxide"],
    "Durée de vie / fonction des globules rouges": [r"red blood cell", r"red cell",
        r"\brbcs?\b", r"lifespan", r"fraction of cells", r"band 3"],
    "Hospitalisation / urgences": [r"hospital", r"admission", r"emergency",
        r"acute care", r"\bed\b"],
    "Qualité de vie / symptômes": [r"quality of life", r"fatigue", r"questionnaire",
        r"pedsql", r"satisfaction"],
    "Greffe / survie": [r"engraftment", r"survival", r"graft", r"chimerism", r"rejection"],
    "Cerveau / vaisseaux": [r"stroke", r"transcranial", r"doppler", r"velocity",
        r"cerebral", r"infarct"],
    "Cognition": [r"\biq\b", r"wechsler", r"\bwais\b", r"\bwasi\b", r"cognition",
        r"executive", r"neurocognitive"],
    "Inflammation / paroi des vaisseaux": [r"vcam", r"selectin", r"c-reactive",
        r"interleukin", r"endothelial", r"microvascular", r"inkt", r"microparticle",
        r"adhesion", r"cytokine"],
    "Organes (rein, poumon, cœur, rate)": [r"renal", r"kidney", r"gfr", r"albumin",
        r"pulmonary", r"lung", r"cardiac", r"myocardial", r"fibrosis", r"spleen"],
    "Sécurité / effets indésirables": [r"adverse", r"safety", r"toxicit", r"tolerab",
        r"malignanc", r"myelosuppress", r"bleeding", r"hemorrhag", r"ulcer", r"platelet"],
    "Pharmacologie (taux sanguins)": [r"pharmacokinetic", r"concentration", r"\bauc\b",
        r"cmax"],
    "Nutrition / métabolisme": [r"energy expenditure", r"25 \(oh\)d", r"vitamin d"],
    "Comportement, éducation, faisabilité": [r"feasib", r"adherence", r"recruitment",
        r"self-efficacy", r"knowledge", r"behavioral", r"acceptab", r"\bpdc\b",
        r"completed follow", r"completing"],
    "Diagnostic": [r"sensitivity", r"specificity", r"predictive value"],
}

# Vrai si au moins un critère de l'étude contient un des mots-clés
def a_theme(liste, mots):
    return any(re.search(m, c.lower()) for c in liste for m in mots)

# Nombre d'études par thème
for theme, mots in themes.items():
    n = res["Criteres_principaux"].apply(lambda l: a_theme(l, mots)).sum()
    print(f"{theme} - {n}")

# Études sans aucun thème reconnu
tous_les_mots = [m for mots in themes.values() for m in mots]
sans_theme = res[res["Criteres_principaux"].apply(lambda l: not a_theme(l, tous_les_mots))]
print("\nÉtudes sans thème reconnu :", len(sans_theme))

# Leurs critères, pour compléter si besoin
for liste in sans_theme["Criteres_principaux"].head(40):
    print("-", " | ".join(liste)[:130])

In [ ]:
# On AJOUTE des mots-clés aux thèmes existants
themes["Douleur / crises"] += [r"opioid", r"analgesic", r"morphine", r"\bmme\b"]
themes["Hospitalisation / urgences"] += [r"length of stay", r"boarding"]
themes["Cerveau / vaisseaux"] += [r"tamv", r"central nervous system"]
themes["Organes (rein, poumon, cœur, rate)"] += [r"\bacr\b", r"trjv", r"exercise",
    r"6 minute walk", r"respiratory", r"breath"]
themes["Greffe / survie"] += [r"transplant", r"stem cell", r"mortality"]
themes["Sécurité / effets indésirables"] += [r"toxic"]
themes["Pharmacologie (taux sanguins)"] += [r"minimum effective dose", r"half-life",
    r"bioavailab"]
themes["Comportement, éducation, faisabilité"] += [r"decisional", r"\boption\b",
    r"app use", r"lessons", r"self-management", r"activation", r"promis"]

# Nouveaux thèmes
themes["Complications aiguës (syndrome thoracique, priapisme)"] = [r"acute chest", r"priapi"]
themes["Fer (surcharge)"] = [r"iron", r"transferrin", r"ferritin"]
themes["Infections et vaccins"] = [r"malaria", r"vaccine", r"antibody response",
    r"pneumococcal", r"immunoglobulin", r"infected", r"tested"]
themes["Transfusion"] = [r"anti-d", r"alloimmun", r"red cell units"]

# Nombre d'études par thème
for theme, mots in themes.items():
    n = res["Criteres_principaux"].apply(lambda l: a_theme(l, mots)).sum()
    print(f"{theme} - {n}")

# Études sans thème
tous_les_mots = [m for mots in themes.values() for m in mots]
sans_theme = res[res["Criteres_principaux"].apply(lambda l: not a_theme(l, tous_les_mots))]
print("\nÉtudes sans thème reconnu :", len(sans_theme))
for liste in sans_theme["Criteres_principaux"]:
    print("-", " | ".join(liste)[:130])

In [ ]:
# Derniers mots-clés
themes["Organes (rein, poumon, cœur, rate)"] += [r"splenic"]
themes["Pharmacologie (taux sanguins)"] += [r"glutamine/glutamate"]
themes["Complications aiguës (syndrome thoracique, priapisme)"] += [r"ventilator"]

# Regroupement des thèmes en familles
familles_criteres = {
    "Événements cliniques et vécu du patient": ["Douleur / crises",
        "Hospitalisation / urgences", "Qualité de vie / symptômes",
        "Complications aiguës (syndrome thoracique, priapisme)", "Cognition"],
    "Marqueurs biologiques et d'organes": ["Hémoglobine / hémolyse",
        "Durée de vie / fonction des globules rouges", "Inflammation / paroi des vaisseaux",
        "Organes (rein, poumon, cœur, rate)", "Cerveau / vaisseaux", "Fer (surcharge)",
        "Nutrition / métabolisme"],
    "Greffe": ["Greffe / survie"],
    "Sécurité et pharmacologie": ["Sécurité / effets indésirables",
        "Pharmacologie (taux sanguins)"],
    "Faisabilité et comportement": ["Comportement, éducation, faisabilité"],
    "Autres (infections, transfusion, diagnostic)": ["Infections et vaccins",
        "Transfusion", "Diagnostic"],
}

# Familles de critères d'UNE étude
def familles_etude_criteres(liste):
    trouvees = []
    for famille, liste_themes in familles_criteres.items():
        if any(a_theme(liste, themes[t]) for t in liste_themes):
            trouvees.append(famille)
    return trouvees if trouvees else ["Non précisé"]

res["Familles_criteres"] = res["Criteres_principaux"].apply(familles_etude_criteres)

# Nombre d'études par famille, et part sur 257
total = len(res)
c = Counter(f for liste in res["Familles_criteres"] for f in liste)
for famille, n in c.most_common():
    print(f"{famille} - {n} ({n / total:.0%})")

# Études qui ne mesurent AUCUN événement clinique ni vécu du patient
clinique = "Événements cliniques et vécu du patient"
sans_clinique = res["Familles_criteres"].apply(lambda l: clinique not in l)
print("\nÉtudes sans critère clinique ni vécu :", sans_clinique.sum(), f"({sans_clinique.mean():.0%})")

# Croisement famille x phase
res["Phase"] = res["Phases"].fillna("Sans phase")
expl = res.explode("Familles_criteres")
print("\n--- Par phase ---")
print(pd.crosstab(expl["Familles_criteres"], expl["Phase"]))

# Croisement famille x type de financeur
print("\n--- Par financeur ---")
print(pd.crosstab(expl["Familles_criteres"], expl["Funder Type"]))

In [ ]:
# Groupes de phases plus lisibles
def groupe_phase(p):
    if p in ["EARLY_PHASE1", "PHASE1"]:
        return "Phase 1"
    if p in ["PHASE1|PHASE2", "PHASE2"]:
        return "Phase 2 (dont 1-2)"
    if p in ["PHASE2|PHASE3", "PHASE3", "PHASE4"]:
        return "Phase 3 et 4 (dont 2-3)"
    return "Sans phase"

res["Groupe_phase"] = res["Phases"].apply(groupe_phase)

# Version stricte : critère clinique ou vécu du patient seulement
clinique = "Événements cliniques et vécu du patient"
res["Clinique_strict"] = res["Familles_criteres"].apply(lambda l: clinique in l)

# Version large : on ajoute les études de greffe (borne haute)
res["Clinique_large"] = res["Familles_criteres"].apply(
    lambda l: clinique in l or "Greffe" in l)

# Part des études avec un critère clinique, par groupe de phase
par_phase = res.groupby("Groupe_phase").agg(
    Etudes=("NCT Number", "count"),
    Strict=("Clinique_strict", "mean"),
    Large=("Clinique_large", "mean"))
par_phase["Strict"] = (par_phase["Strict"] * 100).round(0)
par_phase["Large"] = (par_phase["Large"] * 100).round(0)
print(par_phase)

# Même calcul par type de financeur
par_financeur = res.groupby("Funder Type").agg(
    Etudes=("NCT Number", "count"),
    Strict=("Clinique_strict", "mean"),
    Large=("Clinique_large", "mean"))
par_financeur["Strict"] = (par_financeur["Strict"] * 100).round(0)
par_financeur["Large"] = (par_financeur["Large"] * 100).round(0)
print(par_financeur)

# Totaux globaux
print("\nStrict :", f"{res['Clinique_strict'].mean():.0%}")
print("Large :", f"{res['Clinique_large'].mean():.0%}")

In [ ]:
# Part de critères cliniques (strict) par phase ET par financeur
tab = res.groupby(["Groupe_phase", "Funder Type"]).agg(
    Etudes=("NCT Number", "count"),
    Strict=("Clinique_strict", "mean"))
tab["Strict"] = (tab["Strict"] * 100).round(0)
print(tab)

In [ ]:
# Dossier de sortie (il existe déjà)
sortie = "../data_propre"

# 1. Toutes les études, colonnes utiles seulement
colonnes_df = ["NCT Number", "Study Title", "Study Status", "Study Results", "Study Type",
    "Phases", "Funder Type", "Sponsor", "Start Date", "Enrollment",
    "Pays", "Regions", "Familles_medicaments"]
df[colonnes_df].to_json(f"{sortie}/etudes.json", orient="records", force_ascii=False)

# 2. Études arrêtées ou retirées, avec motif et catégories
colonnes_arretes = ["NCT Number", "Study Status", "Motif_arret", "Categories_arret",
    "Regions", "Familles_medicaments", "Funder Type"]
arretes[colonnes_arretes].to_json(f"{sortie}/arretees.json", orient="records", force_ascii=False)

# 3. Études avec résultats, avec critères principaux et familles
colonnes_res = ["NCT Number", "Study Status", "Groupe_phase", "Funder Type",
    "Criteres_principaux", "Familles_criteres", "Clinique_strict", "Clinique_large",
    "Familles_medicaments", "Regions"]
res[colonnes_res].to_json(f"{sortie}/avec_resultats.json", orient="records", force_ascii=False)

# Vérification : on relit les fichiers et on compte les lignes
for nom in ["etudes", "arretees", "avec_resultats"]:
    verif = pd.read_json(f"{sortie}/{nom}.json")
    print(nom, "-", len(verif), "lignes,", len(verif.columns), "colonnes")

In [ ]:
# Les études avec résultats qui testent un "nouveau médicament ciblé"
cible = res[res["Familles_medicaments"].apply(lambda l: "Nouveaux médicaments ciblés" in l)]
print("Études concernées :", len(cible))

# Les noms des médicaments d'une étude (lignes "DRUG:" de la colonne Interventions)
def noms_medicaments(texte):
    if pd.isna(texte):
        return ""
    noms = [i.replace("DRUG:", "").strip() for i in texte.split("|") if i.startswith("DRUG:")]
    return ", ".join(noms)[:60]

# Pour chaque étude : un en-tête, puis une ligne par bras déclaré
for _, ligne in cible.iterrows():
    nct = ligne["NCT Number"]
    with open(os.path.join(dossier, f"{nct}.json"), encoding="utf-8") as f:
        donnees = json.load(f)

    effets = donnees.get("resultsSection", {}).get("adverseEventsModule", {})
    bras = effets.get("eventGroups", [])

    print(f"\n{nct} | {ligne['Groupe_phase']} | {ligne['Study Status']} | "
          f"{noms_medicaments(ligne['Interventions'])}")

    if not bras:
        print("   (aucun bras déclaré dans les effets indésirables)")
    for b in bras:
        titre = (b.get("title") or "?")[:40]
        graves = b.get("seriousNumAffected")
        risque = b.get("seriousNumAtRisk")
        deces = b.get("deathsNumAffected")
        print(f"   - {titre} | graves : {graves}/{risque} | décès : {deces}")

In [ ]:
# Études comparables : nom du médicament, début des titres des bras médicament, début des titres des bras placebo
comparables = {
    "NCT04218084": ("Voxelotor (enfants)", ["Voxelotor"], ["Placebo"]),
    "NCT03036813": ("Voxelotor (HOPE)", ["Voxelotor 900", "Voxelotor 1500"], ["Placebo"]),
    "NCT05561140": ("Voxelotor (phase en aveugle)", ["Voxelotor, Blinded"], ["Placebo, Blinded"]),
    "NCT04474314": ("IMR-687", ["Higher Dose IMR-687", "Lower Dose IMR-687"], ["Placebo"]),
    "NCT03401112": ("IMR-687", ["All IMR-687"], ["All Placebo"]),
    "NCT02187003": ("Rivipansel", ["Rivipansel Treatment Arm"], ["Placebo Treatment Arm"]),
    "NCT03814746": ("Crizanlizumab", ["Crizanlizumab (SEG101) at 5.0", "Crizanlizumab (SEG101) at 7.5"], ["Placebo"]),
    "NCT04935879": ("Inclacumab", ["Inclacumab"], ["Placebo"]),
    "NCT04927247": ("Inclacumab", ["Inclacumab 30"], ["Placebo"]),
    "NCT00131508": ("Glutamine", ["Glutamine"], ["Placebo"]),
    "NCT00586209": ("L-glutamine", ["L-glutamine"], ["Placebo"]),
    "NCT01179217": ("L-glutamine", ["L-glutamine"], ["100% Maltodextrin"]),
    "NCT00125788": ("L-glutamine", ["Investigational Product"], ["Placebo"]),
    "NCT02114203": ("PDE9i (PF-04447943)", ["PF-04447943 5 mg", "PF-04447943 25 mg"], ["Placebo"]),
}

# Additionne les bras dont le titre commence par un des débuts donnés
def somme_bras(groupes, debuts):
    graves = risque = deces = nb_bras = 0
    deces_connus = True
    for g in groupes:
        titre = g.get("title") or ""
        if any(titre.startswith(d) for d in debuts):
            nb_bras += 1
            graves += g.get("seriousNumAffected") or 0
            risque += g.get("seriousNumAtRisk") or 0
            if g.get("deathsNumAffected") is None:
                deces_connus = False   # None = non déclaré, pas zéro
            else:
                deces += g["deathsNumAffected"]
    return graves, risque, (deces if deces_connus else None), nb_bras

lignes = []
for nct, (medicament, debuts_med, debuts_pla) in comparables.items():
    with open(os.path.join(dossier, f"{nct}.json"), encoding="utf-8") as f:
        donnees = json.load(f)
    groupes = donnees["resultsSection"]["adverseEventsModule"].get("eventGroups", [])

    g_m, r_m, d_m, n_m = somme_bras(groupes, debuts_med)
    g_p, r_p, d_p, n_p = somme_bras(groupes, debuts_pla)

    lignes.append({
        "Étude": nct,
        "Médicament": medicament,
        "Bras (méd./plac.)": f"{n_m}/{n_p}",
        "Graves médicament": f"{g_m}/{r_m}",
        "% méd.": round(100 * g_m / r_m) if r_m else None,
        "Graves placebo": f"{g_p}/{r_p}",
        "% plac.": round(100 * g_p / r_p) if r_p else None,
        "Décès méd.": d_m,
        "Décès plac.": d_p,
    })

tableau = pd.DataFrame(lignes)
print(tableau.to_string(index=False))

In [ ]:
# Titre, âge et effectif des 14 études comparables
pd.set_option("display.max_colwidth", 90)
verif = df[df["NCT Number"].isin(comparables.keys())][
    ["NCT Number", "Study Title", "Age", "Enrollment", "Phases", "Study Status"]]
print(verif.to_string(index=False))

In [ ]:
import math

# Étiquettes corrigées d'après les titres de tes données
etiquettes = {
    "NCT04218084": "Voxelotor - Doppler transcrânien, enfants",
    "NCT03036813": "Voxelotor - HOPE",
    "NCT05561140": "Voxelotor - ulcères de jambe",
    "NCT04474314": "IMR-687 - phase 2, adultes",
    "NCT03401112": "IMR-687 - phase 2, adultes HbSS/Sb0",
    "NCT02187003": "Rivipansel - crise vaso-occlusive",
    "NCT03814746": "Crizanlizumab - 2 doses vs placebo",
    "NCT04935879": "Inclacumab - crises vaso-occlusives",
    "NCT04927247": "Inclacumab - réhospitalisation",
    "NCT00131508": "Glutamine - petite étude",
    "NCT00586209": "L-glutamine - petite étude",
    "NCT01179217": "L-glutamine - phase 3",
    "NCT00125788": "L-glutamine - phase 2",
    "NCT02114203": "PDE9i - phase 1",
}

lignes = []
for nct, (medicament, debuts_med, debuts_pla) in comparables.items():
    with open(os.path.join(dossier, f"{nct}.json"), encoding="utf-8") as f:
        donnees = json.load(f)
    groupes = donnees["resultsSection"]["adverseEventsModule"].get("eventGroups", [])

    g_m, r_m, _, _ = somme_bras(groupes, debuts_med)
    g_p, r_p, _, _ = somme_bras(groupes, debuts_pla)

    p_m, p_p = g_m / r_m, g_p / r_p
    ecart = p_m - p_p
    # Erreur standard de l'écart (méthode simple, approximative)
    se = math.sqrt(p_m * (1 - p_m) / r_m + p_p * (1 - p_p) / r_p)
    bas, haut = ecart - 1.96 * se, ecart + 1.96 * se

    # Trop petit pour conclure ?
    trop_petit = min(r_m, r_p) < 30 or (g_m + g_p) < 10
    if trop_petit:
        verdict = "trop petit pour conclure"
    elif bas > 0:
        verdict = "plus d'effets graves sous médicament"
    elif haut < 0:
        verdict = "moins d'effets graves sous médicament"
    else:
        verdict = "écart compatible avec le hasard"

    lignes.append({
        "Étude": etiquettes[nct],
        "NCT": nct,
        "% méd.": round(100 * p_m),
        "% plac.": round(100 * p_p),
        "Écart (points)": round(100 * ecart, 1),
        "IC 95 %": f"[{100 * bas:.0f} ; {100 * haut:.0f}]",
        "Verdict": verdict,
    })

print(pd.DataFrame(lignes).to_string(index=False))

In [ ]:
# Transforme le tableau et sépare les bornes de l'intervalle en deux colonnes numériques
tab = pd.DataFrame(lignes)
bornes = tab["IC 95 %"].str.extract(r"\[(-?\d+) ; (-?\d+)\]").astype(int)
tab["IC bas"] = bornes[0]
tab["IC haut"] = bornes[1]

# Enregistre dans data_propre
tab.to_json(f"{sortie}/effets_graves_comparaison.json", orient="records", force_ascii=False)

# Vérification : on relit le fichier
verif = pd.read_json(f"{sortie}/effets_graves_comparaison.json")
print("effets_graves_comparaison -", len(verif), "lignes,", len(verif.columns), "colonnes")
print(verif[["Étude", "Écart (points)", "IC bas", "IC haut"]].head(3).to_string(index=False))

In [ ]:
# 1. Part d'études arrêtées par région (parmi les études terminées ou arrêtées)
termines = df[df["Study Status"].isin(["COMPLETED", "TERMINATED", "WITHDRAWN"])].copy()
termines["Arretee"] = termines["Study Status"].isin(["TERMINATED", "WITHDRAWN"])

par_region = termines.explode("Regions").dropna(subset=["Regions"])
t = par_region.groupby("Regions").agg(
    Etudes=("NCT Number", "count"),
    Arretees=("Arretee", "sum"))
t["% arrêtées"] = (100 * t["Arretees"] / t["Etudes"]).round(0)
print("--- Études arrêtées, par région ---")
print(t.sort_values("Etudes", ascending=False))

# 2. Part du motif "Recrutement insuffisant" parmi les études arrêtées, par région
ar = arretes.explode("Regions").dropna(subset=["Regions"]).copy()
ar["Recrutement"] = ar["Categories_arret"].apply(lambda l: "Recrutement insuffisant" in l)
r = ar.groupby("Regions").agg(
    Arretees=("NCT Number", "count"),
    Recrutement=("Recrutement", "sum"))
r["% recrutement"] = (100 * r["Recrutement"] / r["Arretees"]).round(0)
print("\n--- Motif 'Recrutement insuffisant', par région ---")
print(r.sort_values("Arretees", ascending=False))

# 3. Tous les motifs par région (nombre d'études)
ar_motifs = ar.explode("Categories_arret")
print("\n--- Motifs par région ---")
print(pd.crosstab(ar_motifs["Categories_arret"], ar_motifs["Regions"]).to_string())

In [ ]:
pd.set_option("display.width", 200)

# 1. Combien d'études par nombre de régions
df["Nb_regions"] = df["Regions"].str.len()
print("--- Nombre de régions par étude ---")
print(df["Nb_regions"].value_counts().sort_index())

# 2. Part d'arrêtées, études d'UNE SEULE région
termines = df[df["Study Status"].isin(["COMPLETED", "TERMINATED", "WITHDRAWN"])].copy()
termines["Arretee"] = termines["Study Status"].isin(["TERMINATED", "WITHDRAWN"])

mono = termines[termines["Nb_regions"] == 1].copy()
mono["Region"] = mono["Regions"].str[0]
t = mono.groupby("Region").agg(Etudes=("NCT Number", "count"),
                               Arretees=("Arretee", "sum"))
t["% arrêtées"] = (100 * t["Arretees"] / t["Etudes"]).round(0)
print("\n--- Études d'une seule région ---")
print(t.sort_values("Etudes", ascending=False))

# 3. Études multi-régions
multi = termines[termines["Nb_regions"] >= 2]
print("\nÉtudes multi-régions terminées ou arrêtées :", len(multi),
      "| dont arrêtées :", multi["Arretee"].sum())

# 4. Motif "recrutement", études arrêtées d'une seule région
ar_mono = arretes[arretes["Regions"].str.len() == 1].copy()
ar_mono["Region"] = ar_mono["Regions"].str[0]
ar_mono["Recrutement"] = ar_mono["Categories_arret"].apply(
    lambda l: "Recrutement insuffisant" in l)
r = ar_mono.groupby("Region").agg(Arretees=("NCT Number", "count"),
                                  Recrutement=("Recrutement", "sum"))
r["% recrutement"] = (100 * r["Recrutement"] / r["Arretees"]).round(0)
print("\n--- Motif 'Recrutement', études d'une seule région ---")
print(r.sort_values("Arretees", ascending=False))

# 5. Les études arrêtées pour raison de sécurité : sponsor, médicament, régions
secu = arretes[arretes["Categories_arret"].apply(
    lambda l: "Sécurité / risque pour les patients" in l)]
print("\n--- Arrêts pour raison de sécurité ---")
print(secu[["NCT Number", "Sponsor", "Familles_medicaments", "Regions"]].to_string(index=False))

In [ ]:
# 1. Année de début : les 4 premiers caractères de "Start Date" (ex. "2017-03" -> 2017)
df["Annee_debut"] = pd.to_numeric(df["Start Date"].astype(str).str[:4], errors="coerce")
print("Études sans date de début :", df["Annee_debut"].isna().sum())
print("Première et dernière année :", int(df["Annee_debut"].min()), "-", int(df["Annee_debut"].max()))

# 2. Périodes de 5 ans
bornes = [0, 2004, 2009, 2014, 2019, 3000]
noms = ["Avant 2005", "2005-2009", "2010-2014", "2015-2019", "2020 et après"]
df["Periode"] = pd.cut(df["Annee_debut"], bins=bornes, labels=noms)

# 3. Nombre d'études par période
print("\n--- Études par période ---")
print(df["Periode"].value_counts().sort_index())

# 4. Part d'études avec au moins un site en Afrique (parmi celles qui ont un pays)
avec_pays = df[df["Regions"].str.len() > 0].copy()
avec_pays["Site_afrique"] = avec_pays["Regions"].apply(
    lambda l: "Afrique subsaharienne" in l or "Afrique du Nord" in l)
t = avec_pays.groupby("Periode", observed=True).agg(
    Etudes=("NCT Number", "count"),
    Avec_site_Afrique=("Site_afrique", "sum"))
t["% avec site en Afrique"] = (100 * t["Avec_site_Afrique"] / t["Etudes"]).round(0)
print("\n--- Sites en Afrique, par période ---")
print(t)

# 5. Familles de médicaments par période (nombre d'études)
med = df.explode("Familles_medicaments").dropna(subset=["Familles_medicaments"])
choix = ["Hydroxyurée", "Nouveaux médicaments ciblés",
         "Greffe de moelle (conditionnement)", "Mobilisation de cellules souches"]
tab = pd.crosstab(med["Familles_medicaments"], med["Periode"]).reindex(choix)
print("\n--- Familles de médicaments, par période ---")
print(tab)

# Nombre d'études avec au moins un médicament, par période (pour calculer les parts)
avec_med = df[df["Familles_medicaments"].str.len() > 0]
print("\n--- Études avec médicament, par période ---")
print(avec_med["Periode"].value_counts().sort_index())

# 6. Études avec une intervention de type GENETIC, par période
df["Genetique"] = df["Types_intervention"].apply(lambda l: "GENETIC" in l)
print("\n--- Interventions génétiques, par période ---")
print(df.groupby("Periode", observed=True)["Genetique"].sum())

In [ ]:
sortie = "../data_propre"
periodes = ["Avant 2005", "2005-2009", "2010-2014", "2015-2019", "2020 et après"]

# 1. Études qui ont un pays : Afrique seule, ou Afrique avec d'autres régions
avec_pays = df[df["Regions"].str.len() > 0].copy()
afrique = ["Afrique subsaharienne", "Afrique du Nord"]
avec_pays["Site_afrique"] = avec_pays["Regions"].apply(lambda l: any(r in afrique for r in l))
avec_pays["Afrique_seule"] = avec_pays["Regions"].apply(lambda l: all(r in afrique for r in l))
avec_pays["Afrique_et_ailleurs"] = avec_pays["Site_afrique"] & ~avec_pays["Afrique_seule"]

t = avec_pays.groupby("Periode", observed=True).agg(
    Etudes=("NCT Number", "count"),
    Afrique_seule=("Afrique_seule", "sum"),
    Afrique_et_ailleurs=("Afrique_et_ailleurs", "sum"))
t["Total_Afrique"] = t["Afrique_seule"] + t["Afrique_et_ailleurs"]
t["% Afrique"] = (100 * t["Total_Afrique"] / t["Etudes"]).round(0)
print("--- Afrique seule ou avec d'autres régions ---")
print(t)

# 2. Qui mène les études uniquement africaines ?
seules = avec_pays[avec_pays["Afrique_seule"]].sort_values("Annee_debut")
print("\n--- Études dont tous les sites sont en Afrique ---")
print(seules[["NCT Number", "Annee_debut", "Sponsor", "Funder Type", "Study Status"]].to_string(index=False))

# 3. Tableau de l'évolution, enregistré pour le dashboard
avec_med = df[df["Familles_medicaments"].str.len() > 0]
med = df.explode("Familles_medicaments").dropna(subset=["Familles_medicaments"])
df["Genetique"] = df["Types_intervention"].apply(lambda l: "GENETIC" in l)

evol = pd.DataFrame({"Periode": periodes})
evol["Etudes"] = [int((df["Periode"] == p).sum()) for p in periodes]
evol["Etudes_avec_pays"] = [int(t.loc[p, "Etudes"]) for p in periodes]
for col in ["Afrique_seule", "Afrique_et_ailleurs", "Total_Afrique"]:
    evol[col] = [int(t.loc[p, col]) for p in periodes]
evol["Etudes_avec_medicament"] = [int((avec_med["Periode"] == p).sum()) for p in periodes]
for famille in ["Hydroxyurée", "Nouveaux médicaments ciblés",
                "Greffe de moelle (conditionnement)", "Mobilisation de cellules souches"]:
    evol[famille] = [int(((med["Familles_medicaments"] == famille) & (med["Periode"] == p)).sum())
                     for p in periodes]
evol["Genetique"] = [int(df[df["Periode"] == p]["Genetique"].sum()) for p in periodes]

evol.to_json(f"{sortie}/evolution_temps.json", orient="records", force_ascii=False)
print("\n--- Fichier enregistré : evolution_temps.json ---")
print(pd.read_json(f"{sortie}/evolution_temps.json").to_string(index=False))

In [ ]:
# Mots-clés (en minuscules) pour reconnaître l'origine du sponsor
mots_afrique = ["makerere", "lagos", "kinshasa", "muhimbili", "cameroon baptist", "ghana",
    "barau dikko", "angola", "zambia", "global health uganda", "al-neelain", "ain shams",
    "assiut", "beni-suef", "tanta", "cairo", "zagazig", "sohag", "kafrelsheikh"]
mots_hors_afrique = ["vanderbilt", "cincinnati", "indiana university", "loyola", "nevada",
    "duke", "brown university", "new york university", "national heart", "nhlbi",
    "glaxosmithkline", "sanaria", "london school", "liverpool school", "swiss tropical",
    "fondazione"]

def origine_sponsor(nom):
    n = str(nom).lower()
    if any(m in n for m in mots_afrique):
        return "Institution africaine"
    if any(m in n for m in mots_hors_afrique):
        return "Institution hors d'Afrique"
    return "À vérifier"

seules = seules.copy()
seules["Origine_sponsor"] = seules["Sponsor"].apply(origine_sponsor)

# Zone : Afrique du Nord seule, subsaharienne seule, ou les deux
def zone(regions):
    if regions == ["Afrique du Nord"]:
        return "Afrique du Nord"
    if regions == ["Afrique subsaharienne"]:
        return "Afrique subsaharienne"
    return "Les deux zones"
seules["Zone"] = seules["Regions"].apply(zone)

# 1. Origine du sponsor
print("--- Origine du sponsor ---")
print(seules["Origine_sponsor"].value_counts())

# 2. Origine du sponsor selon la zone
print("\n--- Zone x origine du sponsor ---")
print(pd.crosstab(seules["Zone"], seules["Origine_sponsor"]))

# 3. Type de financeur et statut
print("\n--- Type de financeur ---")
print(seules["Funder Type"].value_counts())
print("\n--- Statut ---")
print(seules["Study Status"].value_counts())

# 4. Sponsors à classer à la main
print("\n--- Sponsors à vérifier ---")
print(seules[seules["Origine_sponsor"] == "À vérifier"][["NCT Number", "Sponsor"]].to_string(index=False))

In [ ]:
a_verifier = seules[seules["Origine_sponsor"] == "À vérifier"]

for _, l in a_verifier.iterrows():
    print(l["NCT Number"], "|", l["Sponsor"])
    print("  Titre :", l["Study Title"])
    print("  Collaborateurs :", l["Collaborators"])
    print("  Pays :", l["Pays"])
    print("  Lieux :", str(l["Locations"])[:200])
    print()

In [ ]:
# Classement final des 5 sponsors, d'après les fiches (titre, collaborateurs, lieux)
classement_manuel = {
    "NCT07543289": "Institution africaine",  # collaborateur : Obafemi Awolowo University (Nigeria)
    "NCT06529874": "Institution africaine",  # sponsor individuel, site à Gizeh (Égypte) : probable
}
# NCT01518218, NCT03114137 et NCT03352986 restent "À vérifier"
seules["Origine_sponsor"] = seules.apply(
    lambda l: classement_manuel.get(l["NCT Number"], l["Origine_sponsor"]), axis=1)

print("--- Origine du sponsor ---")
print(seules["Origine_sponsor"].value_counts())

print("\n--- Zone x origine du sponsor ---")
print(pd.crosstab(seules["Zone"], seules["Origine_sponsor"]))

# Quels sponsors hors d'Afrique ? (pour vérifier le décompte)
print("\n--- Sponsors hors d'Afrique ---")
print(seules[seules["Origine_sponsor"] == "Institution hors d'Afrique"]["Sponsor"].value_counts())

# Enregistre dans data_propre
colonnes = ["NCT Number", "Annee_debut", "Sponsor", "Funder Type", "Study Status",
            "Zone", "Origine_sponsor", "Pays", "Study Title"]
seules[colonnes].to_json(f"{sortie}/afrique_seule.json", orient="records", force_ascii=False)
print("\nafrique_seule -", len(pd.read_json(f"{sortie}/afrique_seule.json")), "lignes")

In [ ]:
import os

dossier_ghdx = "../data_brute/ghdx"
fichiers = sorted(os.listdir(dossier_ghdx))
print(fichiers)

# Retrouve le fichier CSV qui contient un code (ex. "ME_2097")
def trouver(code):
    for f in fichiers:
        if code in f and f.lower().endswith(".csv"):
            return os.path.join(dossier_ghdx, f)

scd = {}
for code in ["ME_2097", "ME_2100", "ME_2103", "CAUSE_615"]:
    scd[code] = pd.read_csv(trouver(code))
    print(code, scd[code].shape)

# Premier regard sur un fichier
d = scd["ME_2097"]
print(d.columns.tolist())
for col in ["measure_name", "metric_name", "sex_name", "age_group_name"]:
    print(col, ":", list(d[col].unique())[:15])
print("Années :", d["year_id"].min(), "-", d["year_id"].max())
print("Lieux :", d["location_name"].nunique())

In [ ]:
d = scd["ME_2097"]

# 1. Quelles combinaisons existent pour les naissances (incidence) ?
inc = d[d["measure_name"] == "Incidence"]
print(inc[["age_group_name", "metric_name"]].drop_duplicates().to_string(index=False))

# 2. Naissances 2021, en nombre, les deux sexes
n = d[(d["measure_name"] == "Incidence") & (d["metric_name"] == "Number")
      & (d["sex_name"] == "Both") & (d["year_id"] == 2021)]
print("\nÂges disponibles :", list(n["age_group_name"].unique()))
n = n[n["age_group_name"] == "Birth"]
print("Lieux :", n["location_name"].nunique())

# 3. Les 30 plus grandes valeurs (les regroupements devraient sortir en premier)
print("\n--- 30 plus grandes valeurs ---")
print(n.sort_values("val", ascending=False)[["location_name", "val"]].head(30).to_string(index=False))

# 4. Pays de tes études qui n'ont pas le même nom dans GHDx
noms_ghdx = set(n["location_name"])
absents = sorted(p for p in compteur if p not in noms_ghdx)
print("\n--- Pays de tes études absents de GHDx (sous ce nom) ---")
print(absents)

# 5. Les 3 pays à forte charge sans étude
for p in ["Benin", "Togo", "Equatorial Guinea"]:
    print(p, "présent :", p in noms_ghdx)

In [ ]:
# 1. Naissances 2021 (nombre, les deux sexes) pour chaque génotype, par location_id
def naissances_2021(d):
    n = d[(d["measure_name"] == "Incidence") & (d["metric_name"] == "Number")
          & (d["sex_name"] == "Both") & (d["year_id"] == 2021)
          & (d["age_group_name"] == "Birth")]
    return n[["location_id", "location_name", "val"]]

trois = ["ME_2097", "ME_2100", "ME_2103"]
tab = None
for code in trois:
    n = naissances_2021(scd[code]).rename(columns={"val": code})
    tab = n if tab is None else tab.merge(n[["location_id", code]], on="location_id")
tab["Total"] = tab[trois].sum(axis=1)
print("Lieux :", len(tab))

# 2. TEST : le total mondial retombe-t-il sur ~515 000 ?
print("\n--- Global : les trois génotypes et leur somme ---")
print(tab[tab["location_name"] == "Global"][["location_name"] + trois + ["Total"]].round(0).to_string(index=False))

# 3. Comparaison avec le fichier "Sickle cell disorders" (CAUSE_615)
c = scd["CAUSE_615"]
c = c[(c["measure_name"] == "Incidence") & (c["metric_name"] == "Number")
      & (c["sex_name"] == "Both") & (c["year_id"] == 2021)
      & (c["age_group_name"] == "Birth") & (c["location_name"] == "Global")]
print("\n--- Fichier CAUSE_615, Global ---")
print(c[["location_name", "val"]].round(0).to_string(index=False))

# 4. Recherche des noms de pays qui ne correspondaient pas
recherches = {
    "United States": "United States", "Tanzania": "Tanzania", "Turkey": "rkiye|Turkey",
    "Russia": "Russia", "South Korea": "Korea", "Brunei": "Brunei", "Taiwan": "Taiwan",
    "Côte d’Ivoire": "Ivoire", "Hong Kong": "Hong Kong", "French Guiana": "Guiana",
    "Guadeloupe": "Guadeloupe", "Martinique": "Martinique", "Reunion": "Reunion|Réunion",
}
print("\n--- Noms trouvés dans GHDx ---")
for pays, motif in recherches.items():
    trouves = tab[tab["location_name"].str.contains(motif, case=False, regex=True, na=False)]
    print(pays, "->", trouves[["location_id", "location_name"]].values.tolist()[:6])

In [ ]:
# Correspondance : nom dans tes études -> nom dans GHDx
corr_noms = {
    "United States": "United States of America",
    "Tanzania": "United Republic of Tanzania",
    "Turkey": "Türkiye",
    "Russia": "Russian Federation",
    "South Korea": "Republic of Korea",
    "Brunei": "Brunei Darussalam",
    "Taiwan": "Taiwan (Province of China)",
    "Côte d’Ivoire": "Côte d'Ivoire",
}

total_mondial = tab.loc[tab["location_name"] == "Global", "Total"].iloc[0]
nb_etudes_pays = int((df["Pays"].str.len() > 0).sum())
print("Naissances mondiales :", round(total_mondial), "| Études avec pays :", nb_etudes_pays)

# Un pays par ligne : naissances (GHDx) et nombre d'études
lignes = []
for pays, nb in compteur.items():
    nom = corr_noms.get(pays, pays)
    m = tab[tab["location_name"] == nom]
    if len(m) > 1:
        print("Plusieurs lieux pour", pays, ":", m["location_id"].tolist())
    naiss = m["Total"].iloc[0] if len(m) >= 1 else None
    lignes.append({"Pays": pays, "Naissances": naiss, "Etudes": nb})

# Pays à forte charge sans étude
for pays in ["Benin", "Togo", "Equatorial Guinea"]:
    if pays not in compteur:
        m = tab[tab["location_name"] == pays]
        lignes.append({"Pays": pays, "Naissances": m["Total"].iloc[0], "Etudes": 0})

comp = pd.DataFrame(lignes)
comp["% naissances"] = (100 * comp["Naissances"] / total_mondial).round(2)
comp["% études"] = (100 * comp["Etudes"] / nb_etudes_pays).round(2)

print("\n--- 25 premiers pays par naissances ---")
print(comp.sort_values("Naissances", ascending=False).head(25).round(0).to_string(index=False))

print("\n--- Pays sans estimation GHDx ---")
print(comp[comp["Naissances"].isna()][["Pays", "Etudes"]].to_string(index=False))

# Les lieux GHDx qui ne sont pas dans tes études : top 40, pour repérer ce qui est un pays ou non
deja = set(corr_noms.get(p, p) for p in compteur) | {"Benin", "Togo", "Equatorial Guinea"}
reste = tab[~tab["location_name"].isin(deja)]
print("\n--- 40 plus grands lieux GHDx absents de tes études ---")
print(reste.sort_values("Total", ascending=False).head(40)[["location_id", "location_name", "Total"]].round(0).to_string(index=False))

In [ ]:
# 1. Prévalence 2021 : personnes vivant avec la maladie (tous âges, les deux sexes)
def prevalence_2021(d):
    p = d[(d["measure_name"] == "Prevalence") & (d["metric_name"] == "Number")
          & (d["sex_name"] == "Both") & (d["year_id"] == 2021)
          & (d["age_group_name"] == "All Ages")]
    return p[["location_id", "val"]]

prev = None
for code in trois:
    p = prevalence_2021(scd[code]).rename(columns={"val": code})
    prev = p if prev is None else prev.merge(p, on="location_id")
prev["Prevalence"] = prev[trois].sum(axis=1)
tab2 = tab.merge(prev[["location_id", "Prevalence"]], on="location_id")

# TEST : l'article annonce environ 7,74 millions de personnes en 2021
prev_mondiale = tab2.loc[tab2["location_name"] == "Global", "Prevalence"].iloc[0]
print("Personnes vivant avec la maladie, monde, 2021 :", round(prev_mondiale))

# 2. Pays sans étude repérés dans GHDx, avec leur région (mon classement)
sans_etude = {
    "Benin": "Afrique subsaharienne", "Togo": "Afrique subsaharienne",
    "Equatorial Guinea": "Afrique subsaharienne", "Niger": "Afrique subsaharienne",
    "Guinea": "Afrique subsaharienne", "Madagascar": "Afrique subsaharienne",
    "Chad": "Afrique subsaharienne", "Central African Republic": "Afrique subsaharienne",
    "Congo": "Afrique subsaharienne", "Liberia": "Afrique subsaharienne",
    "Libya": "Afrique du Nord", "Haiti": "Caraïbes et Amérique latine",
    "Iran (Islamic Republic of)": "Moyen-Orient", "Afghanistan": "Asie et Océanie",
}

# 3. Table finale : un pays par ligne
lignes = []
for pays, nb in compteur.items():
    nom = corr_noms.get(pays, pays)
    m = tab2[tab2["location_name"] == nom]
    lignes.append({"Pays": pays, "Region": pays_vers_region.get(pays),
                   "Naissances": m["Total"].iloc[0] if len(m) else None,
                   "Prevalence": m["Prevalence"].iloc[0] if len(m) else None,
                   "Etudes": nb})
for nom, region in sans_etude.items():
    m = tab2[tab2["location_name"] == nom]
    lignes.append({"Pays": nom, "Region": region, "Naissances": m["Total"].iloc[0],
                   "Prevalence": m["Prevalence"].iloc[0], "Etudes": 0})
comp2 = pd.DataFrame(lignes)

comp2["% naissances"] = (100 * comp2["Naissances"] / total_mondial).round(2)
comp2["% prévalence"] = (100 * comp2["Prevalence"] / prev_mondiale).round(2)
comp2["% études"] = (100 * comp2["Etudes"] / nb_etudes_pays).round(2)

# 4. Contrôles
cols = ["Pays", "Etudes", "% naissances", "% prévalence", "% études"]
print("\n--- 20 premiers pays par personnes vivant avec la maladie ---")
print(comp2.sort_values("Prevalence", ascending=False).head(20)[cols].to_string(index=False))

print("\n--- Quelques pays de comparaison ---")
choix = ["United States", "France", "United Kingdom", "Brazil", "Egypt", "Saudi Arabia"]
print(comp2[comp2["Pays"].isin(choix)][cols].to_string(index=False))

zero = comp2[comp2["Etudes"] == 0]
print("\nPays sans étude repérés :", len(zero),
      "| naissances :", round(zero["Naissances"].sum()),
      f"({zero['% naissances'].sum():.1f} % du monde)")

# 5. Enregistrement pour le dashboard
comp2.to_json(f"{sortie}/pays_naissances_etudes.json", orient="records", force_ascii=False)
print("pays_naissances_etudes -", len(pd.read_json(f"{sortie}/pays_naissances_etudes.json")), "lignes")

In [ ]:
# Prévalence mondiale 2021 : chaque génotype, leur somme, et le fichier CAUSE_615
def global_prev(d):
    p = d[(d["measure_name"] == "Prevalence") & (d["metric_name"] == "Number")
          & (d["sex_name"] == "Both") & (d["year_id"] == 2021)
          & (d["age_group_name"] == "All Ages") & (d["location_name"] == "Global")]
    return p[["val", "lower", "upper"]]

somme = 0
for code in trois:
    g = global_prev(scd[code])
    print(code, g.round(0).values.tolist())
    somme += g["val"].iloc[0]
print("Somme des trois génotypes :", round(somme))

print("CAUSE_615 :", global_prev(scd["CAUSE_615"]).round(0).values.tolist())

# Les âges disponibles pour la prévalence dans CAUSE_615
c = scd["CAUSE_615"]
print(c[c["measure_name"] == "Prevalence"]["age_group_name"].unique())

In [ ]:
c = scd["CAUSE_615"]

# Valeur 2021 (nombre, les deux sexes) pour une mesure et un âge donnés
def valeur(c, mesure, age, nom_colonne):
    v = c[(c["measure_name"] == mesure) & (c["metric_name"] == "Number")
          & (c["sex_name"] == "Both") & (c["year_id"] == 2021)
          & (c["age_group_name"] == age)]
    return v[["location_id", "val"]].rename(columns={"val": nom_colonne})

prev_c = valeur(c, "Prevalence", "All Ages", "Prev_cause")
naiss_c = valeur(c, "Incidence", "Birth", "Naiss_cause")

# Retrouve le location_id de chaque pays de la table (en gardant les noms corrigés)
ids = {}
for pays in comp2["Pays"]:
    nom = corr_noms.get(pays, pays)
    m = tab2[tab2["location_name"] == nom]
    ids[pays] = m["location_id"].iloc[0] if len(m) else None
comp2["location_id"] = comp2["Pays"].map(ids)

# On retire d'anciennes colonnes si la cellule est relancée, puis on fusionne
comp2 = comp2.drop(columns=[x for x in ["Prev_cause", "Naiss_cause"] if x in comp2.columns])
comp2 = comp2.merge(prev_c, on="location_id", how="left").merge(naiss_c, on="location_id", how="left")

# Totaux mondiaux selon CAUSE_615 (Global = location_id 1)
prev_glob = prev_c.loc[prev_c["location_id"] == 1, "Prev_cause"].iloc[0]
naiss_glob = naiss_c.loc[naiss_c["location_id"] == 1, "Naiss_cause"].iloc[0]
print("Prévalence mondiale (CAUSE_615) :", round(prev_glob))
print("Naissances mondiales (CAUSE_615) :", round(naiss_glob))

# Nouvelles parts, et écart avec la version précédente (en points de pourcentage)
comp2["% prévalence (CAUSE_615)"] = (100 * comp2["Prev_cause"] / prev_glob).round(2)
comp2["Ecart prévalence (points)"] = (comp2["% prévalence (CAUSE_615)"] - comp2["% prévalence"]).round(2)
comp2["% naissances (CAUSE_615)"] = (100 * comp2["Naiss_cause"] / naiss_glob).round(2)
comp2["Ecart naissances (points)"] = (comp2["% naissances (CAUSE_615)"] - comp2["% naissances"]).round(2)

print("\nÉcart maximal prévalence :", comp2["Ecart prévalence (points)"].abs().max(), "points")
print("Écart maximal naissances :", comp2["Ecart naissances (points)"].abs().max(), "points")

cols = ["Pays", "Etudes", "% naissances (CAUSE_615)", "% prévalence", "% prévalence (CAUSE_615)", "% études"]
print("\n--- 12 premiers pays par prévalence (CAUSE_615) ---")
print(comp2.sort_values("Prev_cause", ascending=False).head(12)[cols].to_string(index=False))

choix = ["United States", "France", "Egypt", "Saudi Arabia", "Brazil"]
print("\n--- Pays de comparaison ---")
print(comp2[comp2["Pays"].isin(choix)][cols].to_string(index=False))

# Enregistre la version mise à jour
comp2.to_json(f"{sortie}/pays_naissances_etudes.json", orient="records", force_ascii=False)
print("\npays_naissances_etudes -", len(pd.read_json(f"{sortie}/pays_naissances_etudes.json")), "lignes")

In [ ]:
# 1. Combien de lignes, combien de pays différents ?
print("Lignes :", len(comp2), "| Pays différents :", comp2["Pays"].nunique())

# 2. Les pays qui apparaissent plus d'une fois
doublons = comp2[comp2.duplicated(subset=["Pays"], keep=False)]
print("\n--- Pays en double ---")
print(doublons[["Pays", "location_id", "Etudes", "Prev_cause", "Naiss_cause"]]
      .sort_values("Pays").to_string(index=False))

# 3. Le problème vient-il des fichiers sources ?
print("\nlocation_id en double dans prev_c :", prev_c["location_id"].duplicated().sum())
print("location_id en double dans naiss_c :", naiss_c["location_id"].duplicated().sum())
dup_ids = prev_c[prev_c["location_id"].duplicated(keep=False)]
print(dup_ids.sort_values("location_id").to_string(index=False))

In [ ]:
# 1. Les lieux en double dans naiss_c : les valeurs sont-elles identiques ?
dup = naiss_c[naiss_c["location_id"].duplicated(keep=False)]
verif = dup.groupby("location_id")["Naiss_cause"].nunique()
print("Lieux en double :", dup["location_id"].nunique())
print("Lieux dont les valeurs diffèrent entre les doublons :", int((verif > 1).sum()))

# 2. Si aucune différence, on supprime les doublons de la table
if (verif > 1).sum() == 0:
    comp2 = comp2.drop_duplicates(subset=["Pays"]).reset_index(drop=True)
    print("\nOK : doublons supprimés")

    # 3. Contrôles
    print("Lignes :", len(comp2), "| Pays différents :", comp2["Pays"].nunique())
    print("Somme des études par pays :", int(comp2["Etudes"].sum()),
          "| attendu :", sum(compteur.values()))
    print("Somme des % naissances (CAUSE_615) :", round(comp2["% naissances (CAUSE_615)"].sum(), 1))
    print("Somme des % prévalence (CAUSE_615) :", round(comp2["% prévalence (CAUSE_615)"].sum(), 1))

    # 4. Enregistrement du fichier corrigé
    comp2.to_json(f"{sortie}/pays_naissances_etudes.json", orient="records", force_ascii=False)
    print("pays_naissances_etudes -", len(pd.read_json(f"{sortie}/pays_naissances_etudes.json")), "lignes")
else:
    print("\nATTENTION : certains doublons ont des valeurs différentes. Rien n'a été supprimé.")
    print(dup.sort_values("location_id").to_string(index=False))

In [ ]:
afrique = ["Afrique subsaharienne", "Afrique du Nord"]
def site_afrique(l):
    return any(r in afrique for r in l)

# 1. Familles de traitements x présence d'un site en Afrique
base = df[df["Regions"].str.len() > 0].copy()
base["Site_afrique"] = base["Regions"].apply(site_afrique)

exp = base.explode("Familles_medicaments").dropna(subset=["Familles_medicaments"])
exp = exp[exp["Familles_medicaments"] != "Placebo / contrôle"]
t = exp.groupby("Familles_medicaments").agg(
    Etudes=("NCT Number", "count"), Avec_Afrique=("Site_afrique", "sum"))
t["% avec site en Afrique"] = (100 * t["Avec_Afrique"] / t["Etudes"]).round(0)
print("--- Familles de traitements et sites en Afrique ---")
print(t.sort_values("Etudes", ascending=False))

# Interventions génétiques
base["Genetique"] = base["Types_intervention"].apply(lambda l: "GENETIC" in l)
g = base[base["Genetique"]]
print("\nInterventions génétiques (avec pays) :", len(g), "| avec site en Afrique :", int(g["Site_afrique"].sum()))

# 2. Études avec résultats : combien ont un site en Afrique ?
r2 = res[res["Regions"].str.len() > 0].copy()
r2["Site_afrique"] = r2["Regions"].apply(site_afrique)
print("\nÉtudes avec résultats et un pays :", len(r2), "| avec site en Afrique :", int(r2["Site_afrique"].sum()))

# 3. Taille médiane des études d'une seule région (colonne Enrollment)
une = base[base["Regions"].str.len() == 1].copy()
une["Region"] = une["Regions"].str[0]
m = une.groupby("Region").agg(
    Etudes=("NCT Number", "count"),
    Enrollment_renseigne=("Enrollment", "count"),
    Mediane=("Enrollment", "median"))
print("\n--- Taille médiane, études d'une seule région ---")
print(m.sort_values("Etudes", ascending=False))

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 55)

afrique = ["Afrique subsaharienne", "Afrique du Nord"]
base = df[df["Regions"].str.len() > 0].copy()
base["Site_afrique"] = base["Regions"].apply(lambda l: any(r in afrique for r in l))
afr = base[base["Site_afrique"]].copy()

# Noms des médicaments d'une étude (lignes "DRUG:")
def noms_med(texte):
    if pd.isna(texte):
        return ""
    return ", ".join(i.replace("DRUG:", "").strip() for i in texte.split("|") if i.startswith("DRUG:"))
afr["Medicaments"] = afr["Interventions"].apply(noms_med)
cols = ["NCT Number", "Annee_debut", "Sponsor", "Phases", "Study Status", "Study Results", "Medicaments"]

# 1. Hydroxyurée et nouveaux médicaments, études avec un site en Afrique
for fam in ["Hydroxyurée", "Nouveaux médicaments ciblés"]:
    sous = afr[afr["Familles_medicaments"].apply(lambda l: fam in l)].sort_values("Annee_debut")
    print(f"\n=== {fam} : {len(sous)} études avec un site en Afrique ===")
    print(sous[cols].to_string(index=False))

# 2. Parmi les nouveaux médicaments, combien sont du voxelotor ?
nouv = afr[afr["Familles_medicaments"].apply(lambda l: "Nouveaux médicaments ciblés" in l)]
vox = nouv["Interventions"].str.lower().str.contains("voxelotor|gbt440|oxbryta", na=False)
print("\nNouveaux médicaments avec site en Afrique :", len(nouv), "| dont voxelotor :", int(vox.sum()))

# 3. Les interventions génétiques avec un site en Afrique
gen = afr[afr["Types_intervention"].apply(lambda l: "GENETIC" in l)]
print("\n=== Interventions génétiques avec site en Afrique ===")
print(gen[["NCT Number", "Annee_debut", "Sponsor", "Study Status", "Study Title"]].to_string(index=False))

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

afrique = ["Afrique subsaharienne", "Afrique du Nord"]
df["Site_afrique"] = df["Regions"].apply(lambda l: any(r in afrique for r in l))
texte = (df["Study Title"].fillna("") + " " + df["Interventions"].fillna("")
         + " " + df["Brief Summary"].fillna("")).str.lower()
cols = ["NCT Number", "Annee_debut", "Sponsor", "Study Status", "Study Title"]

# 1. Greffe et thérapie génique, par mots-clés (et non plus par type d'intervention)
motif = (r"transplant|stem cell|haploidentical|bone marrow|gene therap|gene-therap|lentivir|"
         r"crispr|exagamglogene|exa-cel|ctx001|lovotibeglogene|bb1111|reni-cel|edit-301|gene edit")
c1 = texte.str.contains(motif, regex=True)
print("Greffe / thérapie génique :", int(c1.sum()), "| avec site en Afrique :", int((c1 & df["Site_afrique"]).sum()))
print(df[c1 & df["Site_afrique"]][cols].to_string(index=False))

# 2. Médecines traditionnelles ou à base de plantes
motif2 = (r"\bfaca\b|fagara|zanthox|calotropis|drepanostat|niprisan|cassava|manioc|"
          r"herbal|traditional|phytomedic|medicinal plant|plant extract")
c2 = texte.str.contains(motif2, regex=True)
print("\nMédecines à base de plantes :", int(c2.sum()))
print(df[c2][cols + ["Pays"]].to_string(index=False))

# 3. Les études qui ont un site au Gabon
gab = df[df["Pays"].apply(lambda l: "Gabon" in l)]
print("\nGabon :", len(gab), "études")
print(gab[cols + ["Pays"]].to_string(index=False))

In [ ]:
cibles = list(comparables.keys()) + ["NCT01966731", "NCT01976416", "NCT03806452",
                                     "NCT03634488", "NCT05451940"]
titres = df.set_index("NCT Number")["Study Title"]

# Lit les critères PRINCIPAUX d'une étude : valeurs par bras et analyses statistiques
def lire_primaires(nct):
    with open(os.path.join(dossier, f"{nct}.json"), encoding="utf-8") as f:
        d = json.load(f)
    mesures = d.get("resultsSection", {}).get("outcomeMeasuresModule", {}).get("outcomeMeasures", [])
    sortie = []
    for m in mesures:
        if m.get("type") != "PRIMARY":
            continue
        groupes = {g["id"]: g.get("title", "?") for g in m.get("groups", [])}
        valeurs = []
        for classe in m.get("classes", []):
            for cat in classe.get("categories", []):
                for me in cat.get("measurements", []):
                    valeurs.append((groupes.get(me.get("groupId"), "?")[:32],
                                    me.get("value"), me.get("spread")))
        analyses = [(a.get("pValue"), a.get("statisticalMethod"), a.get("paramType"),
                     a.get("paramValue"), a.get("ciLowerLimit"), a.get("ciUpperLimit"))
                    for a in m.get("analyses", [])]
        sortie.append({"titre": m.get("title"), "delai": m.get("timeFrame"),
                       "unite": m.get("unitOfMeasure"), "type_param": m.get("paramType"),
                       "valeurs": valeurs, "analyses": analyses})
    return sortie

for nct in cibles:
    print("\n" + "=" * 100)
    print(nct, "|", str(titres.get(nct, ""))[:90])
    for m in lire_primaires(nct)[:3]:
        print("  Critère :", (m["titre"] or "")[:110])
        print("  Délai :", (m["delai"] or "")[:60], "| Unité :", m["unite"], "| Type :", m["type_param"])
        for v in m["valeurs"][:8]:
            print("     ", v)
        for a in m["analyses"][:2]:
            print("     analyse (p, méthode, type, valeur, IC bas, IC haut) :", a)

In [ ]:
def lire_mesures(nct, type_mesure="SECONDARY"):
    with open(os.path.join(dossier, f"{nct}.json"), encoding="utf-8") as f:
        d = json.load(f)
    mesures = d.get("resultsSection", {}).get("outcomeMeasuresModule", {}).get("outcomeMeasures", [])
    sortie = []
    for m in mesures:
        if m.get("type") != type_mesure:
            continue
        groupes = {g["id"]: g.get("title", "?") for g in m.get("groups", [])}
        valeurs = []
        for classe in m.get("classes", []):
            for cat in classe.get("categories", []):
                for me in cat.get("measurements", []):
                    valeurs.append((groupes.get(me.get("groupId"), "?")[:30], me.get("value")))
        analyses = [(a.get("pValue"), a.get("paramType"), a.get("paramValue"),
                     a.get("ciLowerLimit"), a.get("ciUpperLimit")) for a in m.get("analyses", [])]
        sortie.append({"titre": m.get("title"), "delai": m.get("timeFrame"),
                       "unite": m.get("unitOfMeasure"), "valeurs": valeurs, "analyses": analyses})
    return sortie

for nct in ["NCT01966731", "NCT01976416"]:
    mesures = lire_mesures(nct)
    print("\n" + "=" * 100)
    print(nct, "|", len(mesures), "critères secondaires")
    for m in mesures[:12]:
        print("\n  Critère :", (m["titre"] or "")[:110])
        print("  Délai :", (m["delai"] or "")[:60], "| Unité :", m["unite"])
        for v in m["valeurs"][:8]:
            print("     ", v)
        for a in m["analyses"][:1]:
            print("     analyse (p, type, valeur, IC bas, IC haut) :", a)

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

afrique = ["Afrique subsaharienne", "Afrique du Nord"]
df["Site_afrique"] = df["Regions"].apply(lambda l: any(r in afrique for r in l))
texte = (df["Study Title"].fillna("") + " " + df["Interventions"].fillna("")).str.lower()

composants = {
    "Antibiotique préventif (pénicilline)": r"penicillin|prophylactic antibiotic|amoxicillin",
    "Vaccins": r"vaccin|pneumococcal|13vpnc|pcv13|ppsv23|immuniz",
    "Paludisme (prévention)": r"malaria|chemoprevent|chemoprophyla|antimalarial|artemether|piperaquine|proguanil|sulfadoxine|bed net",
    "Dépistage / diagnostic": r"newborn screen|neonatal screen|point-of-care|point of care|rapid test|screening",
    "Acide folique": r"folic acid|folate",
    "Transfusion": r"transfusion|erythrocytapheresis|red cell exchange",
}

for nom, motif in composants.items():
    c = texte.str.contains(motif, regex=True)
    print(f"{nom} - {int(c.sum())} études | avec site en Afrique : {int((c & df['Site_afrique']).sum())}")

cols = ["NCT Number", "Annee_debut", "Sponsor", "Study Status", "Study Title"]
for nom in ["Antibiotique préventif (pénicilline)", "Vaccins", "Paludisme (prévention)", "Dépistage / diagnostic"]:
    c = texte.str.contains(composants[nom], regex=True) & df["Site_afrique"]
    print(f"\n=== {nom} : études avec site en Afrique ===")
    print(df[c][cols].to_string(index=False))

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

afrique = ["Afrique subsaharienne", "Afrique du Nord"]
df["Site_afrique"] = df["Regions"].apply(lambda l: any(r in afrique for r in l))
texte = (df["Study Title"].fillna("") + " " + df["Interventions"].fillna("")).str.lower()

motif = (r"genetic counsel|premarital|pre-marital|carrier screen|trait screen|sickle cell trait|"
         r"awareness|health education|educational|knowledge|community health|stigma")
c = texte.str.contains(motif, regex=True)
print("Études conseil / sensibilisation / porteurs :", int(c.sum()),
      "| avec site en Afrique :", int((c & df["Site_afrique"]).sum()))

cols = ["NCT Number", "Annee_debut", "Sponsor", "Study Status", "Study Title"]
print("\n=== Avec site en Afrique ===")
print(df[c & df["Site_afrique"]][cols].to_string(index=False))

print("\n=== Les 15 autres (hors Afrique), pour comparer ===")
print(df[c & ~df["Site_afrique"]][cols].head(15).to_string(index=False))

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 75)

afr_mots = ["africa", "mozambique", "nigeria", "ghana", "kenya", "uganda", "tanzania", "zambia",
    "malawi", "zimbabwe", "rwanda", "ethiopia", "cameroon", "congo", "gabon", "angola", "senegal",
    "mali", "burkina", "benin", "togo", "ivory coast", "côte d", "cote d", "sierra leone",
    "liberia", "guinea", "niger", "chad", "madagascar", "sudan", "egypt", "tunisia", "morocco",
    "algeria", "libya", "south africa", "nairobi", "lagos", "kampala", "dar es salaam",
    "kinshasa", "accra", "abidjan"]
motif = r"\b(?:" + "|".join(afr_mots) + r")\b"

sans_pays = df[df["Pays"].str.len() == 0].copy()
t = (sans_pays["Study Title"].fillna("") + " " + sans_pays["Brief Summary"].fillna("") + " "
     + sans_pays["Sponsor"].fillna("") + " " + sans_pays["Collaborators"].fillna("")).str.lower()
sans_pays["Mots_afrique"] = t.str.findall(motif).apply(lambda l: sorted(set(l)))
trouves = sans_pays[sans_pays["Mots_afrique"].str.len() > 0]

print("Études sans pays :", len(sans_pays), "| avec une mention africaine :", len(trouves))
print(trouves[["NCT Number", "Annee_debut", "Sponsor", "Study Status", "Mots_afrique", "Study Title"]]
      .to_string(index=False))

In [ ]:
# Pays déduits du titre pour 3 études sans site renseigné
ajouts = {"NCT07719972": "Mozambique", "NCT05837871": "Nigeria", "NCT03293641": "Kenya"}
pays_vers_region["Mozambique"] = "Afrique subsaharienne"

def maj_pays(l):
    return [ajouts[l["NCT Number"]]] if l["NCT Number"] in ajouts else l["Pays"]
def maj_regions(l):
    if l["NCT Number"] in ajouts:
        return [pays_vers_region[ajouts[l["NCT Number"]]]]
    return l["Regions"]

df["Pays"] = df.apply(maj_pays, axis=1)
df["Regions"] = df.apply(maj_regions, axis=1)

# Recomptage
compteur = Counter(p for liste in df["Pays"] for p in liste)
nb_etudes_pays = int((df["Pays"].str.len() > 0).sum())
afrique = ["Afrique subsaharienne", "Afrique du Nord"]
df["Site_afrique"] = df["Regions"].apply(lambda l: any(r in afrique for r in l))
ssa = df["Regions"].apply(lambda l: "Afrique subsaharienne" in l)

print("Études avec pays :", nb_etudes_pays)
print("Études avec site en Afrique :", int(df["Site_afrique"].sum()),
      f"({100 * df['Site_afrique'].sum() / nb_etudes_pays:.1f} %)")
print("Études avec site en Afrique subsaharienne :", int(ssa.sum()),
      f"({100 * ssa.sum() / nb_etudes_pays:.1f} %)")
print("Mozambique :", compteur["Mozambique"], "| Nigeria :", compteur["Nigeria"],
      "| Kenya :", compteur["Kenya"])

# Réenregistre le fichier des études
df[colonnes_df].to_json(f"{sortie}/etudes.json", orient="records", force_ascii=False)
print("etudes -", len(pd.read_json(f"{sortie}/etudes.json")), "lignes")

In [ ]:
# --- 1. Table pays : naissances, prévalence, études (version à jour) ---
lignes = []
for pays, nb in compteur.items():
    nom = corr_noms.get(pays, pays)
    m = tab2[tab2["location_name"] == nom]
    lignes.append({"Pays": pays, "Region": pays_vers_region.get(pays), "Etudes": nb,
                   "location_id": m["location_id"].iloc[0] if len(m) else None})
for nom, region in sans_etude.items():
    if nom not in compteur:
        m = tab2[tab2["location_name"] == nom]
        lignes.append({"Pays": nom, "Region": region, "Etudes": 0,
                       "location_id": m["location_id"].iloc[0]})
comp3 = pd.DataFrame(lignes)
comp3 = comp3.merge(prev_c.drop_duplicates("location_id"), on="location_id", how="left")
comp3 = comp3.merge(naiss_c.drop_duplicates("location_id"), on="location_id", how="left")
comp3 = comp3.rename(columns={"Prev_cause": "Prevalence", "Naiss_cause": "Naissances"})

prev_glob = prev_c.loc[prev_c["location_id"] == 1, "Prev_cause"].iloc[0]
naiss_glob = naiss_c.loc[naiss_c["location_id"] == 1, "Naiss_cause"].iloc[0]
comp3["% naissances"] = (100 * comp3["Naissances"] / naiss_glob).round(2)
comp3["% prévalence"] = (100 * comp3["Prevalence"] / prev_glob).round(2)
comp3["% études"] = (100 * comp3["Etudes"] / nb_etudes_pays).round(2)

print("Lignes :", len(comp3), "| Pays différents :", comp3["Pays"].nunique())
print("Somme des études :", int(comp3["Etudes"].sum()), "| attendu :", sum(compteur.values()))
print("Somme % naissances :", round(comp3["% naissances"].sum(), 1),
      "| % prévalence :", round(comp3["% prévalence"].sum(), 1))
choix = ["Nigeria", "India", "Democratic Republic of the Congo", "Kenya", "Mozambique", "United States"]
print(comp3[comp3["Pays"].isin(choix)][["Pays", "Etudes", "% naissances", "% prévalence", "% études"]]
      .to_string(index=False))
comp3.drop(columns="location_id").to_json(f"{sortie}/pays_naissances_etudes.json",
                                          orient="records", force_ascii=False)

# --- 2. Études uniquement africaines (version à jour) ---
classement_manuel["NCT07719972"] = "Institution hors d'Afrique"  # St. Jude (États-Unis), Mozambique
classement_manuel["NCT05837871"] = "Institution africaine"       # Ahmadu Bello University Teaching Hospital
avec_pays = df[df["Regions"].str.len() > 0].copy()
seules = avec_pays[avec_pays["Regions"].apply(lambda l: all(r in afrique for r in l))].copy()
seules["Origine_sponsor"] = seules["Sponsor"].apply(origine_sponsor)
seules["Origine_sponsor"] = seules.apply(
    lambda l: classement_manuel.get(l["NCT Number"], l["Origine_sponsor"]), axis=1)
seules["Zone"] = seules["Regions"].apply(zone)
print("\nÉtudes uniquement africaines :", len(seules))
print(pd.crosstab(seules["Zone"], seules["Origine_sponsor"]))
colonnes = ["NCT Number", "Annee_debut", "Sponsor", "Funder Type", "Study Status",
            "Zone", "Origine_sponsor", "Pays", "Study Title"]
seules[colonnes].to_json(f"{sortie}/afrique_seule.json", orient="records", force_ascii=False)

# --- 3. Régions des études avec résultats (version à jour) ---
res["Regions"] = res["NCT Number"].map(df.set_index("NCT Number")["Regions"])
res[colonnes_res].to_json(f"{sortie}/avec_resultats.json", orient="records", force_ascii=False)
print("\nFichiers réenregistrés : pays_naissances_etudes, afrique_seule, avec_resultats")

In [ ]:
pd.set_option("display.width", 220)

print("=== 1. Ensemble ===")
print("Études :", len(df), "| avec pays :", nb_etudes_pays, "| avec résultats :", len(res))
print("Arrêtées ou retirées :", len(arretes))
print(df["Study Status"].value_counts().head(6).to_string())

print("\n=== 2. Où ===")
n_ghdx = naiss_c.drop_duplicates("location_id").set_index("location_id")["Naiss_cause"]
p_ghdx = prev_c.drop_duplicates("location_id").set_index("location_id")["Prev_cause"]
print("Afrique subsaharienne (GBD) - naissances :", round(100 * n_ghdx[166] / naiss_glob, 1),
      "% | prévalence :", round(100 * p_ghdx[166] / prev_glob, 1), "%")
afrique = ["Afrique subsaharienne", "Afrique du Nord"]
df["Site_afrique"] = df["Regions"].apply(lambda l: any(r in afrique for r in l))
ssa = df["Regions"].apply(lambda l: "Afrique subsaharienne" in l)
print("Études avec site en Afrique :", int(df["Site_afrique"].sum()),
      f"({100 * df['Site_afrique'].sum() / nb_etudes_pays:.1f} %)")
print("Études avec site en Afrique subsaharienne :", int(ssa.sum()),
      f"({100 * ssa.sum() / nb_etudes_pays:.1f} %)")
trio = comp3[comp3["Pays"].isin(["Nigeria", "India", "Democratic Republic of the Congo"])]
print("Nigeria + Inde + RDC : prévalence", round(trio["% prévalence"].sum(), 1),
      "% | études (plafond)", round(trio["% études"].sum(), 1), "%")
zero = comp3[comp3["Etudes"] == 0]
print("Pays sans étude repérés :", len(zero), "| naissances :", round(zero["% naissances"].sum(), 1),
      "% | prévalence :", round(zero["% prévalence"].sum(), 1), "%")

print("\n=== 3. Qui ===")
print(pd.crosstab(seules["Zone"], seules["Origine_sponsor"]))
hors = seules[(seules["Zone"] == "Afrique subsaharienne")
              & (seules["Origine_sponsor"] == "Institution hors d'Afrique")]
print(hors["Sponsor"].value_counts().head(5).to_string())
print(df["Funder Type"].value_counts().to_string())

print("\n=== 4. Pourquoi les essais s'arrêtent ===")
total = len(arretes)
cnt = Counter(cat for liste in arretes["Categories_arret"] for cat in liste)
for cat, n in cnt.most_common():
    print(f"{cat} - {n} ({n / total:.0%})")

print("\n=== 5. Ce que mesurent les essais avec résultats ===")
print("Critère clinique principal : strict", f"{res['Clinique_strict'].mean():.0%}",
      "| large", f"{res['Clinique_large'].mean():.0%}")
print(res.groupby("Groupe_phase")[["Clinique_strict", "Clinique_large"]].mean().mul(100).round(0))

print("\n=== 6. Familles de traitements et sites en Afrique ===")
base = df[df["Regions"].str.len() > 0].copy()
exp = base.explode("Familles_medicaments").dropna(subset=["Familles_medicaments"])
exp = exp[exp["Familles_medicaments"] != "Placebo / contrôle"]
t = exp.groupby("Familles_medicaments").agg(Etudes=("NCT Number", "count"),
                                            Avec_Afrique=("Site_afrique", "sum"))
t["%"] = (100 * t["Avec_Afrique"] / t["Etudes"]).round(0)
print(t.sort_values("Etudes", ascending=False))

print("\n=== 7. Évolution dans le temps ===")
print(pd.read_json(f"{sortie}/evolution_temps.json").to_string(index=False))